# Stage A1 — Extract per-layer activations

**Experiment A — Representation Convergence (GPT-2 vs Pythia-160M)**

Hypothesis: *well-trained independent models share representations up to a
linear transformation.* The models were chosen from different labs,
architectures, tokenizers, and training corpora (OpenAI/WebText vs
EleutherAI/The Pile), both trained from scratch — so any alignment found
must have been discovered independently by each training run.


## What this stage does
Runs the **same 2,000 WikiText sentences** through both models and saves the
mean-pooled hidden state of **every layer**. Also extracts a random-weights
copy of the Pythia architecture as a control baseline — it calibrates how
much similarity comes from architecture + input statistics alone, so the
trained-minus-random gap is the measured effect of *learning*.

## Expected output
`activations.npz` with `A_layers [L_A, N, d_A]`, `B_layers [L_B, N, d_B]`,
`R_layers` (random baseline). Runtime ~2-5 min on GPU.


In [ ]:
# Storage setup — where stage artifacts (.npz, .png) are read/written.
# Each stage reads the previous stage's output from DATA_DIR.
#
# Option 1 (default): current directory. Works if you run ALL stages in
# the SAME runtime/session. In Colab, a new notebook = a new VM, so files
# from a previous notebook are gone.
#
# Option 2 (Colab, persistent): mount Google Drive and point DATA_DIR
# there — artifacts survive across notebooks and sessions:
#
# from google.colab import drive
# drive.mount('/content/drive')
# os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

import os
os.environ.setdefault("DATA_DIR", ".")
print("DATA_DIR =", os.path.abspath(os.environ["DATA_DIR"]))

In [ ]:
# Install dependencies (once)
# !pip install torch transformers datasets numpy

In [ ]:
# Configuration and imports
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
DATA_DIR.mkdir(parents=True, exist_ok=True)

import torch
from transformers import AutoModel, AutoTokenizer, AutoConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_SAMPLES = 2000
MAX_LEN = 64
BATCH = 32

MODEL_A = "gpt2"
MODEL_B = "EleutherAI/pythia-160m"

In [ ]:
# Functions
def load_sentences(n):
    from datasets import load_dataset
    try:
        ds = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1",
                          split="validation")
    except Exception:
        # fallback: any English corpus works, content just needs diversity
        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1",
                          split="validation")
    sents = [t.strip() for t in ds["text"] if len(t.strip()) > 100]
    return sents[:n]


@torch.no_grad()
def extract(model, tokenizer, sentences):
    """Returns [n_layers, n_samples, hidden_dim] — mean-pooled over tokens."""
    model.eval().to(DEVICE)
    all_layers = None
    for i in range(0, len(sentences), BATCH):
        batch = sentences[i : i + BATCH]
        enc = tokenizer(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_LEN,
        ).to(DEVICE)
        out = model(**enc, output_hidden_states=True)
        mask = enc["attention_mask"].unsqueeze(-1)  # [B, T, 1]
        # hidden_states: tuple of [B, T, D], one per layer (incl. embeddings)
        pooled = [
            ((h * mask).sum(1) / mask.sum(1)).float().cpu().numpy()
            for h in out.hidden_states
        ]
        pooled = np.stack(pooled)  # [L, B, D]
        all_layers = pooled if all_layers is None else np.concatenate(
            [all_layers, pooled], axis=1
        )
        if (i // BATCH) % 10 == 0:
            print(f"  {i}/{len(sentences)}")
    return all_layers


def main():
    print("Loading sentences...")
    sentences = load_sentences(N_SAMPLES)
    print(f"{len(sentences)} sentences")

    print(f"\nExtracting from {MODEL_A}...")
    tok_a = AutoTokenizer.from_pretrained(MODEL_A)
    tok_a.pad_token = tok_a.eos_token
    model_a = AutoModel.from_pretrained(MODEL_A)
    A = extract(model_a, tok_a, sentences)
    del model_a
    torch.cuda.empty_cache()

    print(f"\nExtracting from {MODEL_B}...")
    tok_b = AutoTokenizer.from_pretrained(MODEL_B)
    tok_b.pad_token = tok_b.eos_token
    model_b = AutoModel.from_pretrained(MODEL_B)
    B = extract(model_b, tok_b, sentences)
    del model_b
    torch.cuda.empty_cache()

    print("\nExtracting random-weights baseline (untrained Pythia arch)...")
    cfg = AutoConfig.from_pretrained(MODEL_B)
    rand_model = AutoModel.from_config(cfg)
    R = extract(rand_model, tok_b, sentences)

    np.savez_compressed(
        str(DATA_DIR / "activations.npz"), A_layers=A, B_layers=B, R_layers=R
    )
    print(f"\nSaved: A {A.shape}, B {B.shape}, R {R.shape}")

In [ ]:
# Run the extraction
main()